<a href="https://colab.research.google.com/github/SaloneJJ/FHT-Structures-ans-Sperner-Enumerations/blob/main/FHT_Enumeration_Symmetry_and_Ultrametric.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pynauty networkx


In [ ]:
# FHT Enumeration, Symmetry & Canonical Metric Analysis v3.0.0 by SALONE Jean-Jacques 2026
# jean-jacques.salone@univ-antilles.fr

from itertools import combinations, permutations
from collections import defaultdict
from pynauty import Graph, certificate, autgrp


# ==========================================
# HELPER: FORMATTING EDGES TO SET NOTATION
# ==========================================

def edges_to_set_string(edges, v):
    """Converts binary integer bitmasks into readable mathematical set notation (1-indexed)."""
    formatted_edges = []
    for e in edges:
        verts = []
        temp = e
        idx = 1
        while temp:
            if temp & 1:
                verts.append(str(idx))
            temp >>= 1
            idx += 1
        formatted_edges.append("{" + ", ".join(verts) + "}")
    return "E = {" + ", ".join(formatted_edges) + "}"


# ==========================================
# PART 1: S(v) ENUMERATION & SYMMETRY METADATA
# ==========================================

def get_possible_edges(v):
    edges = []
    for r in range(1, v + 1):
        for comb in combinations(range(v), r):
            edge_mask = 0
            for vertex in comb:
                edge_mask |= (1 << vertex)
            edges.append(edge_mask)
    return edges

def is_sperner_extension(current_edges, new_edge):
    for e in current_edges:
        if (new_edge & e) == new_edge or (new_edge & e) == e:
            return False
    return True

def get_primal_adjacency(v, edges):
    adj = {i: set() for i in range(v)}
    for e in edges:
        verts = []
        temp = e
        vertex_idx = 0
        while temp:
            if temp & 1:
                verts.append(vertex_idx)
            temp >>= 1
            vertex_idx += 1
        for i in range(len(verts)):
            for j in range(i + 1, len(verts)):
                u, w = verts[i], verts[j]
                adj[u].add(w)
                adj[w].add(u)
    return adj

def is_connected_bit(v, edges):
    if not edges:
        return False
    adj_list = get_primal_adjacency(v, edges)
    adj_masks = [0] * v
    for u, neighbors in adj_list.items():
        for n in neighbors:
            adj_masks[u] |= (1 << n)

    visited = 1 << 0
    queue = [0]
    while queue:
        curr = queue.pop(0)
        unvisited = adj_masks[curr] & ~visited
        while unvisited:
            bit = unvisited & -unvisited
            next_v = bit.bit_length() - 1
            visited |= bit
            queue.append(next_v)
            unvisited ^= bit

    return visited == ((1 << v) - 1)

def compute_vertex_distance_matrix(v, edges):
    adj_list = get_primal_adjacency(v, edges)
    matrix = [[0] * v for _ in range(v)]
    for i in range(v):
        dist = {i: 0}
        queue = [i]
        while queue:
            curr = queue.pop(0)
            for neighbor in adj_list[curr]:
                if neighbor not in dist:
                    dist[neighbor] = dist[curr] + 1
                    queue.append(neighbor)
        for j in range(v):
            matrix[i][j] = dist.get(j, float('inf'))
    return tuple(tuple(row) for row in matrix)

def canonical_distance_matrix(matrix):
    v = len(matrix)
    if v <= 1:
        return matrix
    best_mat = None
    for perm in permutations(range(v)):
        perm_mat = tuple(tuple(matrix[perm[i]][perm[j]] for j in range(v)) for i in range(v))
        if best_mat is None or perm_mat < best_mat:
            best_mat = perm_mat
    return best_mat

def get_pynauty_graph(v, edges):
    m = len(edges)
    n_nodes = v + m
    adj = {i: [] for i in range(n_nodes)}
    vertex_class = set(range(v))
    edge_class = set(range(v, v + m))

    for idx, e in enumerate(edges):
        edge_node = v + idx
        temp = e
        u = 0
        while temp:
            if temp & 1:
                adj[u].append(edge_node)
                adj[edge_node].append(u)
            temp >>= 1
            u += 1

    return Graph(number_of_vertices=n_nodes, directed=False, adjacency_dict=adj, vertex_coloring=[vertex_class, edge_class])

def extract_metadata(v, edges, structure_type="Sperner (Flat)"):
    g = get_pynauty_graph(v, edges)
    aut_data = autgrp(g)
    aut_size = aut_data[1] * (10 ** aut_data[2])
    orbits = aut_data[3]
    num_orbits = aut_data[4]
    vertex_orbits = orbits[:v]
    fixed_points_count = sum(1 for node_val in vertex_orbits if vertex_orbits.count(node_val) == 1)

    raw_dist = compute_vertex_distance_matrix(v, edges)
    dist_matrix = canonical_distance_matrix(raw_dist)
    diameter = max([d for row in dist_matrix for d in row if d != float('inf')]) if v > 1 else 0
    scale_factor = 1.0 / (diameter + 1)

    return {
        "v": v,
        "type": structure_type,
        "edges": list(edges),
        "set_repr": edges_to_set_string(edges, v),
        "aut_size": aut_size,
        "orbits": num_orbits,
        "inv_points": fixed_points_count,
        "dist_matrix": dist_matrix,
        "diameter": diameter,
        "scale_factor": scale_factor
    }

def compute_S_v_with_metadata(v):
    if v == 1: return 1, [extract_metadata(1, [1], "Sperner (Flat)")]
    if v == 2: return 1, [extract_metadata(2, [3], "Sperner (Flat)")]
    all_edges = get_possible_edges(v)
    seen_certificates = set()
    structures_metadata = []

    def backtrack(edge_index, current_hg):
        if current_hg:
            g = get_pynauty_graph(v, current_hg)
            cert = certificate(g)
            if cert in seen_certificates: return
            seen_certificates.add(cert)
            if is_connected_bit(v, current_hg):
                structures_metadata.append(extract_metadata(v, current_hg, "Sperner (Flat)"))

        for i in range(edge_index, len(all_edges)):
            candidate = all_edges[i]
            if is_sperner_extension(current_hg, candidate):
                current_hg.append(candidate)
                backtrack(i + 1, current_hg)
                current_hg.pop()

    backtrack(0, [])
    return len(structures_metadata), structures_metadata


# ==========================================
# PART 2: I(v) HIERARCHICAL DECOMPOSITION
# ==========================================

def generate_valid_partitions(v):
    if v <= 2: return []
    partitions = []
    def backtrack_part(remaining, max_val, current):
        if remaining == 0:
            if len(current) > 0 and current[0] > 1: partitions.append(tuple(current))
            return
        start = min(remaining, max_val)
        for i in range(start, 0, -1):
            if i < v: backtrack_part(remaining - i, i, current + [i])
    backtrack_part(v, v - 1, [])
    seen, valid = set(), []
    for p in partitions:
        if p[0] > 1 and all(x < v for x in p) and p not in seen:
            seen.add(p); valid.append(p)
    return valid

def compute_I_v_with_metadata(v, all_fht_details):
    if v <= 2: return 0, []
    partitions = generate_valid_partitions(v)
    imbricated_metadata = []
    total_count = 0

    for p in partitions:
        def build_compositions(part_idx, current_sub_fhts_meta, current_vertex_offset):
            nonlocal total_count
            if part_idx == len(p):
                global_edges = []
                offset = 0
                for block_size, sub_fht in zip(p, current_sub_fhts_meta):
                    for edge_mask in sub_fht["edges"]:
                        global_edges.append(edge_mask << offset)
                    offset += block_size
                global_edges.append((1 << v) - 1)
                imbricated_metadata.append(extract_metadata(v, global_edges, "Imbricated (Hierarchical)"))
                total_count += 1
                return
            for sub_fht in all_fht_details[p[part_idx]]:
                build_compositions(part_idx + 1, current_sub_fhts_meta + [sub_fht], current_vertex_offset + p[part_idx])
        build_compositions(0, [], 0)
    return total_count, imbricated_metadata


# ==========================================
# MAIN EXECUTION SCRIPT (INTERACTIVE)
# ==========================================

def fraction_str(val):
    if abs(val - 0.5) < 1e-5: return "1/2"
    if abs(val - 0.3333) < 1e-3: return "1/3"
    if abs(val - 0.25) < 1e-5: return "1/4"
    if abs(val - 1.0) < 1e-5: return "1"
    return f"{val:.3f}"

def mat_to_string(matrix):
    return "\n".join(["    [" + ", ".join([str(x) if x != float('inf') else 'inf' for x in row]) + "]" for row in matrix])

if __name__ == "__main__":
    print("\n" + "="*70)
    print(" FRACTAL HYPER-TREES (FHT) v3.0.0 — INTERACTIVE ANALYTICAL ENGINE")
    print("="*70)

    try:
        user_input = input("Entrez le nombre maximal de sommets (v) [Défaut = 4] : ")
        max_v = int(user_input) if user_input.strip() else 4
    except ValueError:
        max_v = 4
        print("Entrée invalide détectée. Utilisation de la valeur par défaut : v = 4.")

    S, I = {}, {}
    all_fht_details = {}
    global_catalogue = []

    print(f"\n--- Lancement de l'analyse de v = 1 à {max_v} ---\n")

    f_counter = 1
    for v in range(1, max_v + 1):
        if v <= 2:
            s_val = 1
            s_det = [extract_metadata(1, [1], "Sperner (Flat)")] if v == 1 else [extract_metadata(2, [3], "Sperner (Flat)")]
            S[v], s_details = s_val, s_det
        else:
            S[v], s_details = compute_S_v_with_metadata(v)

        I[v], i_details = compute_I_v_with_metadata(v, all_fht_details) if v > 2 else (0, [])

        current_v_details = s_details + i_details
        all_fht_details[v] = current_v_details

        print(f"\n[Dimension v = {v}]")
        print(f"  • Structures Sperner (Plates) S({v}) = {S[v]}")
        print(f"  • Structures Imbriquées I({v}) = {I[v]}")
        print(f"  • Total FHT f({v}) = {S[v] + I[v]}")

        for meta in current_v_details:
            meta["global_id"] = f_counter
            global_catalogue.append(meta)
            print(f"    -> F_{f_counter} [{meta['type']}]: {meta['set_repr']}")
            print(f"       [Symétrie] Aut Size: {meta['aut_size']} | Orbits: {meta['orbits']} | Invariant Points: {meta['inv_points']}")
            f_counter += 1

    # ==========================================
    # GLOBAL CANONICAL METRIC ISOMETRY CLASSES
    # ==========================================
    print("\n" + "="*70)
    print(" GLOBAL CANONICAL METRIC ISOMETRY CLASSES (MACRO-STATES)")
    print("="*70)

    metric_classes = defaultdict(list)
    for fht in global_catalogue:
        signature = (fht["v"], fht["diameter"], fht["dist_matrix"])
        metric_classes[signature].append(fht)

    for class_idx, (sig, members) in enumerate(sorted(metric_classes.items(), key=lambda x: (x[0][0], x[0][1]))):
        v_val, diam, mat = sig
        scale = 1.0 / (diam + 1)
        member_ids = [f"F_{m['global_id']}" for m in members]
        print(f"Macro-State M_{class_idx}: v = {v_val} | Diameter (Δ) = {diam} | Scale (S) = {fraction_str(scale)}")
        print(f"  Members: {', '.join(member_ids)}")
        print(f"  Canonical Distance Matrix:\n{mat_to_string(mat)}")
        print("-" * 50)